# AI Act Q&A: answer evaluation on a GPU

Every candidate model answers the development questions from the passages retrieval finds, and the judge (Gemma 4 31B) grades the answers. The code is the repository [vladzpop11/ai-act-qa](https://github.com/vladzpop11/ai-act-qa); this notebook only runs `scripts/kaggle_run.sh`.

**Settings:** Accelerator **GPU T4 x2** (the judge needs both GPUs' 32 GB), Internet **on**. Run with **Save Version → Save & Run All (Commit)**, so it keeps running with the browser closed. Expected time: about 3 hours (10 min building llama.cpp, ~40 GB of model downloads, answers, then grading).

Results: `/kaggle/working/results` (the version's **Output** tab), including `answers_dev.json`.

In [ ]:
!nvidia-smi

In [ ]:
%cd /tmp
!rm -rf ai-act-qa && git clone -q https://github.com/vladzpop11/ai-act-qa
%cd /tmp/ai-act-qa
!git log -1 --oneline
!pip install -q uv==0.13.1
!uv sync --locked -q
!uv run python -m ai_act_qa.ingest --check && uv run python -m ai_act_qa.parse

In [ ]:
# A failing shell command does not stop a notebook, so its exit status is checked here.
from pathlib import Path

!bash scripts/kaggle_run.sh dev; echo $? > /tmp/kaggle_run_status
status = Path("/tmp/kaggle_run_status").read_text().strip()
if status != "0":
    raise SystemExit(
        f"kaggle_run.sh failed with status {status}: see the output above "
        "or results/run.log in the Output tab"
    )

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("/kaggle/working/results/answers_dev.json").read_text())
keys = ("score", "correct", "grounded", "declines_unanswerable", "cites_expected_passage")
for name, metrics in summary["models"].items():
    print(name, {k: metrics[k] for k in keys})